# tanque-cheio · 04 — previsões finais no teste (M7)

Gera, **numa única sessão e na mesma GPU**, as previsões dos 2 sistemas do relatório
final (D24) no `datasets/test.jsonl` (150 perguntas), formato 5.3:

1. `base-qwen2.5-coder-1.5b-instruct__test.jsonl`: o modelo base, sem treino;
2. `ft-qwen2.5-coder-1.5b-instruct__test.jsonl`: o mesmo base + o adapter LoRA do M6,
   baixado do Hugging Face (`ADAPTER_ID`, repo público) e **fundido** no base
   (`merge_and_unload`) antes de gerar, para a latência ser comparável à do base.

Os dois usam o prompt único (`tanque.prompting.build_prompt`) pelo chat template,
decodificação gulosa, `max_new_tokens=256`, fp16, lote de 1. Cada um grava também um
`run-info__<sistema>__test.json` (commits do repositório, do checkpoint e do adapter,
versões, `adapter_merged`, GPU). A lógica está em `tanque.infer.local` (testada sem GPU).

**Por que um notebook à parte do `03_predict`:** o 03 roda um modelo num split; aqui os
dois sistemas precisam sair da mesma sessão, com o mesmo hardware, os mesmos pins de
versão e o checkpoint e o adapter fixados por commit. Juntar isso no 03 encheria a célula
de parâmetros de casos especiais; o teste é rodado uma vez só.

**No Kaggle:** Settings → Accelerator → **GPU T4**; **Internet on**. *Save Version → Save
& Run All*. Tempo estimado: ~15 a 25 min (2 × 150 perguntas a ~3 s, mais downloads).

As métricas não são calculadas aqui: baixe os JSONL e rode `uv run tanque eval score` no
computador (passos em `notebooks/README.md`, seção M7).

In [ ]:
# Parâmetros: edite só esta célula.
REPO_REF = "main"  # branch, tag ou commit do tanque-cheio
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # modelo base (M5)
# Commit do checkpoint usado no M6 (run-info__ft-...__dev.json); None = main do HF.
MODEL_REVISION = "2e1fd397ee46e1388853d2af2c993145b0f1098a"
ADAPTER_ID = "belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora"  # adapter LoRA no HF (público)
ADAPTER_REVISION = "bd8079ff5467c339586bc7fc31f18a19b6509a9e"  # commit do adapter; None = main
MERGE_ADAPTER = True  # funde o LoRA no base antes de gerar (latência justa)
DATASET = "test"  # o relatório final usa o teste, uma vez por sistema
LIMIT = None  # ex.: 3 para um ensaio rápido (os arquivos ganham o sufixo -sample3)
CHAT_TEMPLATE_KWARGS = {}  # Qwen2.5 não precisa de nada
SYSTEM_SUFFIX = ""
DTYPE = "float16"  # a T4 não tem bf16 nativo
DEVICE = "cuda"
MAX_NEW_TOKENS = 256
OUT_DIR = "/kaggle/working"  # vira a saída do notebook no Kaggle
REPO_DIR = "/tmp/tanque-cheio"  # fora de OUT_DIR, para não ir para a saída

In [ ]:
# Instala o pacote `tanque` no REPO_REF (com as versões do M6) e clona o repositório no
# mesmo ref, para ter datasets/, configs/ e data/manifest.json.
import os

# O Kaggle pode dar 2 GPUs T4: uma só, antes de qualquer import do torch, para os dois
# sistemas rodarem na mesma GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess
import sys

REPO_URL = "https://github.com/belasoaress/tanque-cheio"
# As mesmas versões do fine-tuning e das previsões no dev (uv.lock); torch vem da imagem.
PINS = [
    "transformers==5.21.0",
    "peft==0.21.2",
    "accelerate==1.15.0",
]


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(sys.executable, "-m", "pip", "install", "-q", f"tanque @ git+{REPO_URL}@{REPO_REF}", *PINS)
# A imagem do Kaggle traz um torchao antigo (0.10) e o peft se recusa a carregar o
# adapter quando ele está instalado. O projeto não usa torchao: remove.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

In [ ]:
import torch
import transformers
from peft.import_utils import is_torchao_available

print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| visíveis:", torch.cuda.device_count())
else:
    print("Sem GPU: ligue o acelerador GPU T4 em Settings > Accelerator.")

# Falha cedo (antes da inferência) se o peft não puder carregar adapters.
is_torchao_available()  # levanta ImportError se houver um torchao incompatível

In [ ]:
import json
from pathlib import Path

from tanque import config
from tanque.infer import local

# Os dois sistemas do relatório final (D24), nesta ordem, na mesma sessão e GPU.
SYSTEMS = [
    {"adapter_id": None, "adapter_revision": None},  # base-<modelo>
    {"adapter_id": ADAPTER_ID, "adapter_revision": ADAPTER_REVISION},  # ft-<modelo>
]

runs = []
for order, spec in enumerate(SYSTEMS, start=1):
    info = local.run_predictions(
        config.resolve_path(f"datasets/{DATASET}.jsonl"),
        MODEL_ID,
        out_dir=OUT_DIR,
        revision=MODEL_REVISION,
        merge_adapter=MERGE_ADAPTER,
        limit=LIMIT,
        dtype=DTYPE,
        device=DEVICE,
        max_new_tokens=MAX_NEW_TOKENS,
        chat_template_kwargs=CHAT_TEMPLATE_KWARGS,
        system_suffix=SYSTEM_SUFFIX,
        **spec,
    )
    info.update(
        repo_ref=REPO_REF,
        repo_commit=REPO_COMMIT,
        session_order=order,
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    )
    run_info = Path(OUT_DIR) / f"run-info__{Path(info['path']).stem}.json"
    run_info.write_text(json.dumps(info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(info, ensure_ascii=False, indent=2))
    runs.append(info)

assert len({r["hardware"] for r in runs}) == 1, "os dois sistemas precisam do mesmo hardware"
assert runs[1]["adapter_merged"] is MERGE_ADAPTER

In [ ]:
# Primeiras previsões de cada sistema, para conferir o formato e a SQL extraída.
for info in runs:
    print("==", info["system"], info["summary"])
    for line in Path(info["path"]).read_text(encoding="utf-8").splitlines()[:3]:
        p = json.loads(line)
        print(p["id"], p["latency_ms"], "ms |", p["error"] or p["sql"][:200].replace("\n", " "))